In [70]:
!pip install rasterio
!pip install cartopy

In [71]:
import pandas as pd
import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.mask import mask
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import geopandas as gpd
from shapely.geometry import Point
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import ListedColormap, BoundaryNorm
import os


windspeed_files = [
    ('/Users/jackywong1105/Desktop/Project 1/WindReturnMap/downloads/max_wind_speed_2021346N05145.csv','2021346N05145'),
    ('/Users/jackywong1105/Desktop/Project 1/WindReturnMap/downloads/max_wind_speed_2017347N11131.csv','2017347N11131'),
    ('/Users/jackywong1105/Desktop/Project 1/WindReturnMap/downloads/max_wind_speed_2014190N08154.csv','2014190N08154')
]

return_levels_df = pd.read_csv('/Users/jackywong1105/Desktop/Project 1/WindReturnMap/results/all_grid_summaries.csv')
ibtracs_points_path = 'IBTrACS.WP.list.v04r01.points.shp'
ibtracs_lines_path = 'IBTrACS.WP.list.v04r01.lines.shp'

# Path to shapefile
shapefile_path = 'PHL_adm0.shp'


In [72]:
return_levels_df

,longitude_latitude,return_level_1years,return_level_2years,return_level_3years,return_level_4years,return_level_5years,return_level_6years,return_level_7years,return_level_8years,return_level_9years,...,return_level_991years,return_level_992years,return_level_993years,return_level_994years,return_level_995years,return_level_996years,return_level_997years,return_level_998years,return_level_999years,return_level_1000years
0,grid_116.6999999999997897_6.8999999999999417,-inf,-inf,-inf,-inf,8.571836,8.777502,8.922887,9.030961,9.114369,...,9.736376,9.736381,9.736385,9.736390,9.736394,9.736399,9.736403,9.736408,9.736412,9.736417
1,grid_116.7999999999997840_6.8999999999999417,-inf,-inf,-inf,-inf,9.135100,9.214957,9.282475,9.340961,9.392550,...,11.451795,11.452237,11.452678,11.453119,11.453560,11.454000,11.454439,11.454878,11.455317,11.455755
2,grid_116.8999999999997783_7.2999999999999403,-inf,-inf,-inf,-inf,13.523563,13.665945,13.786328,13.890608,13.982590,...,17.654177,17.654965,17.655752,17.656538,17.657323,17.658108,17.658891,17.659674,17.660456,17.661238
3,grid_116.9999999999997726_7.8999999999999382,-inf,-inf,-inf,-inf,13.752244,13.914142,14.051024,14.169597,14.274185,...,18.449000,18.449896,18.450790,18.451684,18.452577,18.453469,18.454360,18.455250,18.456140,18.457028
4,grid_116.9999999999997726_7.9999999999999378,-inf,-inf,-inf,-inf,13.966144,14.107258,14.226568,14.329919,14.421081,...,18.059952,18.060732,18.061512,18.062291,18.063070,18.063847,18.064624,18.065400,18.066175,18.066949
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2394,grid_126.4999999999992326_7.4999999999999396,-inf,-inf,-inf,-inf,4.875013,5.093431,5.278100,5.438068,5.579170,...,11.211463,11.212671,11.213878,11.215084,11.216288,11.217492,11.218694,11.219895,11.221095,11.222293
2395,grid_126.4999999999992326_7.5999999999999392,-inf,-inf,-inf,-inf,5.564046,5.819779,6.035998,6.223295,6.388503,...,12.983040,12.984454,12.985868,12.987279,12.988690,12.990099,12.991506,12.992913,12.994317,12.995721
2396,grid_126.4999999999992326_7.6999999999999389,-inf,-inf,-inf,-inf,6.613711,6.884245,7.112978,7.311115,7.485885,...,14.462083,14.463580,14.465075,14.466569,14.468061,14.469551,14.471040,14.472528,14.474014,14.475498
2397,grid_126.5999999999992269_7.2999999999999403,-inf,-inf,-inf,-inf,9.014808,9.187779,9.334025,9.460708,9.572451,...,14.032832,14.033788,14.034744,14.035699,14.036653,14.037606,14.038558,14.039509,14.040460,14.041409


In [73]:
# Split the longitude_latitude column, removing the 'grid_' prefix
return_levels_df[['Longitude', 'Latitude']] = return_levels_df['longitude_latitude'].str.replace('grid_', '').str.split('_', expand=True)
return_levels_df['Longitude'] = return_levels_df['Longitude'].astype(float)
return_levels_df['Latitude'] = return_levels_df['Latitude'].astype(float)

In [74]:
# Round coordinates to 11 decimal places for matching
return_levels_df['Longitude'] = return_levels_df['Longitude'].round(11)
return_levels_df['Latitude'] = return_levels_df['Latitude'].round(11)

In [75]:
return_levels_df

,longitude_latitude,return_level_1years,return_level_2years,return_level_3years,return_level_4years,return_level_5years,return_level_6years,return_level_7years,return_level_8years,return_level_9years,...,return_level_993years,return_level_994years,return_level_995years,return_level_996years,return_level_997years,return_level_998years,return_level_999years,return_level_1000years,Longitude,Latitude
0,grid_116.6999999999997897_6.8999999999999417,-inf,-inf,-inf,-inf,8.571836,8.777502,8.922887,9.030961,9.114369,...,9.736385,9.736390,9.736394,9.736399,9.736403,9.736408,9.736412,9.736417,116.7,6.9
1,grid_116.7999999999997840_6.8999999999999417,-inf,-inf,-inf,-inf,9.135100,9.214957,9.282475,9.340961,9.392550,...,11.452678,11.453119,11.453560,11.454000,11.454439,11.454878,11.455317,11.455755,116.8,6.9
2,grid_116.8999999999997783_7.2999999999999403,-inf,-inf,-inf,-inf,13.523563,13.665945,13.786328,13.890608,13.982590,...,17.655752,17.656538,17.657323,17.658108,17.658891,17.659674,17.660456,17.661238,116.9,7.3
3,grid_116.9999999999997726_7.8999999999999382,-inf,-inf,-inf,-inf,13.752244,13.914142,14.051024,14.169597,14.274185,...,18.450790,18.451684,18.452577,18.453469,18.454360,18.455250,18.456140,18.457028,117.0,7.9
4,grid_116.9999999999997726_7.9999999999999378,-inf,-inf,-inf,-inf,13.966144,14.107258,14.226568,14.329919,14.421081,...,18.061512,18.062291,18.063070,18.063847,18.064624,18.065400,18.066175,18.066949,117.0,8.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2394,grid_126.4999999999992326_7.4999999999999396,-inf,-inf,-inf,-inf,4.875013,5.093431,5.278100,5.438068,5.579170,...,11.213878,11.215084,11.216288,11.217492,11.218694,11.219895,11.221095,11.222293,126.5,7.5
2395,grid_126.4999999999992326_7.5999999999999392,-inf,-inf,-inf,-inf,5.564046,5.819779,6.035998,6.223295,6.388503,...,12.985868,12.987279,12.988690,12.990099,12.991506,12.992913,12.994317,12.995721,126.5,7.6
2396,grid_126.4999999999992326_7.6999999999999389,-inf,-inf,-inf,-inf,6.613711,6.884245,7.112978,7.311115,7.485885,...,14.465075,14.466569,14.468061,14.469551,14.471040,14.472528,14.474014,14.475498,126.5,7.7
2397,grid_126.5999999999992269_7.2999999999999403,-inf,-inf,-inf,-inf,9.014808,9.187779,9.334025,9.460708,9.572451,...,14.034744,14.035699,14.036653,14.037606,14.038558,14.039509,14.040460,14.041409,126.6,7.3


In [76]:
# Define return period mapping to numeric codes
return_period_mapping = {
    'No Data': 0,
    '< 7-year': 1,
    '7-year': 2,
    '8-year': 3,
    '9-year': 4,
    '10-year': 5,
    '11-year': 6,
    '12-year': 7,
    '13-year': 8,
    '14-year': 9,
    '15-year': 10,
    '16-year': 11,
    '17-year': 12,
    '18-year': 13,
    '19-year': 14,
    '20-year': 15,
    '21-year': 16,
    '22-year': 17,
    '23-year': 18,
    '24-year': 19,
    '25-year': 20,
    '26-year': 21,
    '27-year': 22,
    '28-year': 23,
    '29-year': 24,
    '30-year': 25,
    '50-year': 26,
    '100-year': 27,
    '250-year': 28,
    '250-year or greater': 29
}
# Generate 30 colors from gist_ncar colormap
gist_stern = cm.get_cmap('gist_stern', 30)  # Sample 30 colors
colors = [gist_stern(1 - (i / 29)) for i in range(30)]  # Reverse the colormap

# Create color labels
color_labels = list(return_period_mapping.keys())

# Create color map and normalization
cmap = ListedColormap(colors)
norm = BoundaryNorm(list(range(len(return_period_mapping) + 1)), cmap.N)


/var/folders/6s/rz60r97n7lsgxbhkfswp4qc00000gn/T/ipykernel_30163/1249483630.py:35: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  gist_stern = cm.get_cmap('gist_stern', 30)  # Sample 30 colors


In [77]:
try:
    ibtracs_lines = gpd.read_file(ibtracs_lines_path)
    ibtracs_points = gpd.read_file(ibtracs_points_path)
except Exception as e:
    raise ValueError(f"Error reading IBTrACS files: {e}")


In [78]:
def get_return_period(row):
    max_wind_speed = row['max_wind_speed']
    if pd.isna(max_wind_speed):
        return 'No Data'
    elif max_wind_speed < row['return_level_7years']:
        return '< 7-year'
    elif max_wind_speed < row['return_level_8years']:
        return '7-year'
    elif max_wind_speed < row['return_level_9years']:
        return '8-year'
    elif max_wind_speed < row['return_level_10years']:
        return '9-year'
    elif max_wind_speed < row['return_level_11years']:
        return '10-year'
    elif max_wind_speed < row['return_level_12years']:
        return '11-year'
    elif max_wind_speed < row['return_level_13years']:
        return '12-year'
    elif max_wind_speed < row['return_level_14years']:
        return '13-year'
    elif max_wind_speed < row['return_level_15years']:
        return '14-year'
    elif max_wind_speed < row['return_level_16years']:
        return '15-year'
    elif max_wind_speed < row['return_level_17years']:
        return '16-year'
    elif max_wind_speed < row['return_level_18years']:
        return '17-year'
    elif max_wind_speed < row['return_level_19years']:
        return '18-year'
    elif max_wind_speed < row['return_level_20years']:
        return '19-year'
    elif max_wind_speed < row['return_level_21years']:
        return '20-year'
    elif max_wind_speed < row['return_level_22years']:
        return '21-year'
    elif max_wind_speed < row['return_level_23years']:
        return '22-year'
    elif max_wind_speed < row['return_level_24years']:
        return '23-year'
    elif max_wind_speed < row['return_level_25years']:
        return '24-year'
    elif max_wind_speed < row['return_level_26years']:
        return '25-year'
    elif max_wind_speed < row['return_level_27years']:
        return '26-year'
    elif max_wind_speed < row['return_level_28years']:
        return '27-year'
    elif max_wind_speed < row['return_level_29years']:
        return '28-year'
    elif max_wind_speed < row['return_level_30years']:
        return '29-year'
    elif max_wind_speed < row['return_level_50years']:
        return '30-year'
    elif max_wind_speed < row['return_level_100years']:
        return '50-year'
    elif max_wind_speed < row['return_level_250years']:
        return '100-year'
    elif max_wind_speed < row['return_level_500years']:
        return '250-year'
    else:
        return '250-year or greater'


In [79]:
# Function to create GeoTIFF from CSV
def csv_to_tiff(csv_df, output_tiff, shapefile_path, value_column='Return_Period_Code', resolution=0.18):
    # Read shapefile
    try:
        boundary = gpd.read_file(shapefile_path)
        if boundary.crs != 'EPSG:4326':
            boundary = boundary.to_crs(epsg=4326)
    except Exception as e:
        raise ValueError(f"Error reading shapefile: {e}")

    # Get shapefile bounds
    bounds = boundary.total_bounds  # [minx, miny, maxx, maxy]
    lon_min, lat_min, lon_max, lat_max = bounds
    print(f"Shapefile bounds for {output_tiff}: {bounds}")

    # Verify coordinates
    if not (csv_df['Longitude'].between(lon_min, lon_max).any() and csv_df['Latitude'].between(lat_min, lat_max).any()):
        print(f"Warning: No coordinates in CSV fall within shapefile bounds {bounds}")

    # Convert to GeoDataFrame
    gdf = gpd.GeoDataFrame(
        csv_df,
        geometry=[Point(xy) for xy in zip(csv_df['Longitude'], csv_df['Latitude'])],
        crs="EPSG:4326"
    )

    # Calculate grid dimensions
    cols = int((lon_max - lon_min) / resolution) + 1
    rows = int((lat_max - lat_min) / resolution) + 1

    # Create empty grid
    grid = np.full((rows, cols), np.nan, dtype=np.float32)

    # Populate grid with return period codes
    for _, row in csv_df.iterrows():
        col = int((row['Longitude'] - lon_min) / resolution)
        row_idx = int((lat_max - row['Latitude']) / resolution)
        if 0 <= row_idx < rows and 0 <= col < cols:
            grid[row_idx, col] = row[value_column]

    # Define transform for GeoTIFF
    transform = from_origin(lon_min, lat_max, resolution, resolution)

    # Write temporary TIFF
    temp_tiff = 'temp_output.tiff'
    try:
        with rasterio.open(
            temp_tiff,
            'w',
            driver='GTiff',
            height=rows,
            width=cols,
            count=1,
            dtype=grid.dtype,
            crs='EPSG:4326',
            transform=transform,
            nodata=np.nan,
        ) as dst:
            dst.write(grid, 1)
    except Exception as e:
        raise ValueError(f"Error writing temporary TIFF: {e}")

    # Mask the TIFF with shapefile
    try:
        with rasterio.open(temp_tiff) as src:
            masked_data, masked_transform = mask(src, boundary.geometry, crop=True, nodata=np.nan)
        os.remove(temp_tiff)  # Clean up temporary file
    except Exception as e:
        raise ValueError(f"Error masking TIFF with shapefile: {e}")

    # Write final TIFF
    try:
        with rasterio.open(
            output_tiff,
            'w',
            driver='GTiff',
            height=masked_data.shape[1],
            width=masked_data.shape[2],
            count=1,
            dtype=masked_data.dtype,
            crs='EPSG:4326',
            transform=masked_transform,
            nodata=np.nan,
        ) as dst:
            dst.write(masked_data[0], 1)
    except Exception as e:
        raise ValueError(f"Error writing final TIFF: {e}")

In [80]:
# Function to create PNG from GeoTIFF
def tiff_to_png(tiff_file, output_png, shapefile_path,sid):
    # Read shapefile
    try:
        boundary = gpd.read_file(shapefile_path)
        if boundary.crs != 'EPSG:4326':
            boundary = boundary.to_crs(epsg=4326)
    except Exception as e:
        raise ValueError(f"Error reading shapefile: {e}")

    # Read TIFF
    with rasterio.open(tiff_file) as src:
        data = src.read(1)
        transform = src.transform
        extent = [transform.c, transform.c + transform.a * src.width,
                  transform.f + transform.e * src.height, transform.f]

    # Check for valid data
    if np.all(np.isnan(data)):
        print(f"Warning: No valid data in {tiff_file}. Skipping PNG generation.")
        return

    # Create figure with white background
    fig = plt.figure(figsize=(12, 8), facecolor='white')
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_facecolor('white')

    # Set extent to Philippines region
    ax.set_extent([115, 130, 5, 20], crs=ccrs.PlateCarree())

    # Add geographic features
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.5)

    # Plot data with equal aspect for square cells
    cmap.set_bad('white')  # No-data as white
    im = ax.imshow(
        data,
        cmap=cmap,
        norm=norm,
        extent=extent,
        transform=ccrs.PlateCarree(),
        origin='upper',
        aspect='equal'
    )

    # Overlay shapefile boundary
    boundary.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=1)

    # Plot TC track
    track_line = ibtracs_lines[ibtracs_lines['SID'] == sid]
    track_points = ibtracs_points[ibtracs_points['SID'] == sid]
    if not track_line.empty:
        track_line.plot(ax=ax, color='black', linewidth=0.5, transform=ccrs.PlateCarree())
    '''if not track_points.empty:
        track_points.plot(ax=ax, color='red', marker='o', markersize=5, linestyle='None', transform=ccrs.PlateCarree())'''


    # Add gridlines
    ax.gridlines(draw_labels=True, linestyle='--', alpha=0.5)

    # Add colorbar with selected labels to avoid clutter
    key_ticks = [0, 1, 2, 5, 10, 15, 20, 25, 26, 27, 28, 29]  # No Data, < 7-year, 7, 10, 15, 20, 25, 30, 50, 100, 250, 250+
    cbar = fig.colorbar(im, ax=ax, ticks=key_ticks, orientation='vertical')
    cbar.set_label('Return Period')
    cbar.ax.set_yticklabels([color_labels[i] for i in key_ticks])

    # Set title
    title = f'Return Periods from with TC {sid}'
    ax.set_title(title)

    # Save PNG
    try:
        plt.savefig(output_png, dpi=300, bbox_inches='tight', facecolor='white')
    except Exception as e:
        raise ValueError(f"Error saving PNG: {e}")
    plt.close()

In [81]:
# Process each windspeed CSV
for windspeed_files,sid in windspeed_files:
    try:
        windspeed_df = pd.read_csv(windspeed_files)
    except Exception as e:
        print(f"Error reading {windspeed_files}: {e}")
        continue

    # Round coordinates for matching
    windspeed_df['Longitude'] = windspeed_df['longitude'].round(11)
    windspeed_df['Latitude'] = windspeed_df['latitude'].round(11)

    # Merge with return levels
    merged_df = pd.merge(
        windspeed_df,
        return_levels_df,
        on=['Longitude', 'Latitude'],
        how='left'
    )

    # Determine return periods
    merged_df['Return_Period'] = merged_df.apply(get_return_period, axis=1)

    # Map return periods to numeric codes
    merged_df['Return_Period_Code'] = merged_df['Return_Period'].map(return_period_mapping)

    # Select output columns
    result_df = merged_df[['Longitude', 'Latitude', 'max_wind_speed', 'Return_Period']]

    # Save to CSV
    output_csv = f'return_periods_{os.path.basename(windspeed_files)}'
    try:
        result_df.to_csv(output_csv, index=False)
        print(f"  CSV saved: {output_csv}")
    except Exception as e:
        print(f"Error saving CSV {output_csv}: {e}")

    # Create GeoTIFF
    output_tiff = f'return_periods_{os.path.splitext(os.path.basename(windspeed_files))[0]}.tiff'
    csv_to_tiff(merged_df, output_tiff, shapefile_path, 'Return_Period_Code')

    # Create PNG
    output_png = f'return_periods_{os.path.splitext(os.path.basename(windspeed_files))[0]}.png'
    tiff_to_png(output_tiff, output_png, shapefile_path,sid)

    # Print summary
    print(f"\nProcessed {windspeed_files}:")
    print(f"  CSV saved: {output_csv}")
    print(f"  GeoTIFF saved: {output_tiff}")
    print(f"  PNG saved: {output_png}")
    print(result_df.head())

  CSV saved: return_periods_max_wind_speed_2021346N05145.csv
Shapefile bounds for return_periods_max_wind_speed_2021346N05145.tiff: [116.94999   5.04917 126.59804  19.39111]

Processed /Users/jackywong1105/Desktop/Project 1/WindReturnMap/downloads/max_wind_speed_2021346N05145.csv:
  CSV saved: return_periods_max_wind_speed_2021346N05145.csv
  GeoTIFF saved: return_periods_max_wind_speed_2021346N05145.tiff
  PNG saved: return_periods_max_wind_speed_2021346N05145.png
   Longitude  Latitude  max_wind_speed Return_Period
0      121.4      19.4       16.730963      < 7-year
1      121.5      19.4       16.614357      < 7-year
2      121.9      18.9       15.997801      < 7-year
3      120.8      18.6       13.055245      < 7-year
4      120.9      18.6       12.344164      < 7-year
  CSV saved: return_periods_max_wind_speed_2017347N11131.csv
Shapefile bounds for return_periods_max_wind_speed_2017347N11131.tiff: [116.94999   5.04917 126.59804  19.39111]

Processed /Users/jackywong1105/Deskto